# RAG Pipeline Evaluation with EvalAI

This notebook shows how to **evaluate retrieval-augmented generation (RAG) pipelines** using the EvalAI Python SDK.

You'll learn:
1. Assert RAG output quality — grounding, relevance, hallucination checks
2. Run a test suite against a RAG pipeline
3. Use snapshot testing to detect regressions

In [ ]:
# Install (run once)
# !pip install pauly4010-evalai-sdk

## 1. Simulated RAG Pipeline

We'll create a mock RAG pipeline to demonstrate evaluation patterns. Replace this with your real retriever + generator.

In [ ]:
KNOWLEDGE_BASE = {
    "python": "Python is a high-level programming language created by Guido van Rossum in 1991. It emphasizes code readability and supports multiple paradigms.",
    "rust": "Rust is a systems programming language focused on safety, speed, and concurrency. It was created by Graydon Hoare at Mozilla.",
    "javascript": "JavaScript is a dynamic programming language primarily used for web development. It was created by Brendan Eich in 1995.",
}


def mock_retrieve(query: str) -> list[str]:
    """Simulate a retriever — returns relevant knowledge chunks."""
    query_lower = query.lower()
    return [v for k, v in KNOWLEDGE_BASE.items() if k in query_lower]


def mock_generate(query: str, context: list[str]) -> str:
    """Simulate a generator — produces an answer grounded in context."""
    if not context:
        return "I don't have enough information to answer that question."
    return f"Based on the available information: {context[0]}"


async def rag_pipeline(query: str) -> str:
    """Full RAG pipeline: retrieve then generate."""
    chunks = mock_retrieve(query)
    return mock_generate(query, chunks)


# Quick test
answer = await rag_pipeline("Tell me about Python")
print(answer)

## 2. Assert RAG Output Quality

Check grounding, factual accuracy, hallucination, and structure:

In [ ]:
from evalai_sdk import expect

answer = await rag_pipeline("Tell me about Python")
ground_truth = ["Guido van Rossum", "1991", "high-level"]

results = [
    # Is the answer grounded in known facts?
    expect(answer).to_contain_keywords(["Python", "Guido van Rossum"]),

    # Does it hallucinate facts not in the source?
    expect(answer).to_not_hallucinate(ground_truth),

    # Is it a reasonable length?
    expect(answer).to_have_length(min=20, max=500),

    # No PII leakage?
    expect(answer).to_not_contain_pii(),

    # Professional tone?
    expect(answer).to_be_professional(),
]

for r in results:
    status = "PASS" if r.passed else "FAIL"
    print(f"[{status}] {r.assertion_type}: {r.message or 'OK'}")

print(f"\n{sum(r.passed for r in results)}/{len(results)} checks passed")

## 3. RAG Test Suite

Define multiple test cases to evaluate the pipeline systematically:

In [ ]:
from evalai_sdk import create_test_suite
from evalai_sdk.types import TestSuiteCase, TestSuiteConfig

suite = create_test_suite(
    "rag-quality",
    TestSuiteConfig(
        evaluator=rag_pipeline,
        test_cases=[
            TestSuiteCase(
                name="python-knowledge",
                input="Tell me about Python",
                assertions=[
                    {"type": "contains_keywords", "value": ["Python", "Guido"]},
                    {"type": "has_length", "value": {"min": 20}},
                ],
            ),
            TestSuiteCase(
                name="rust-knowledge",
                input="Tell me about Rust",
                assertions=[
                    {"type": "contains_keywords", "value": ["Rust", "safety"]},
                    {"type": "not_contains_pii"},
                ],
            ),
            TestSuiteCase(
                name="unknown-topic",
                input="Tell me about quantum physics",
                assertions=[
                    {"type": "contains", "value": "don't have enough"},
                ],
            ),
            TestSuiteCase(
                name="javascript-knowledge",
                input="Tell me about JavaScript",
                assertions=[
                    {"type": "contains_keywords", "value": ["JavaScript", "Brendan"]},
                    {"type": "not_contains_pii"},
                ],
            ),
        ],
    ),
)

result = await suite.run()

print(f"Suite: {result.suite_name}")
print(f"Score: {result.score:.2f}")
print(f"Passed: {result.passed_count}/{result.total}\n")

for r in result.results:
    status = "PASS" if r.passed else "FAIL"
    print(f"  [{status}] {r.test_case_name} ({r.duration_ms:.0f}ms)")

## 4. Snapshot Testing

Save a known-good output and compare future runs against it to catch regressions:

In [ ]:
from evalai_sdk import SnapshotManager, compare_with_snapshot

manager = SnapshotManager(snapshot_dir="./snapshots")

# Save baseline output
answer = await rag_pipeline("Tell me about Python")
manager.save("rag-python-answer", {"output": answer, "source": "mock"})
print(f"Saved snapshot: {answer[:80]}...\n")

# Later — compare a new run against the snapshot
new_answer = await rag_pipeline("Tell me about Python")
comparison = manager.compare("rag-python-answer", {"output": new_answer, "source": "mock"})

print(f"Match: {comparison.matches}")
if not comparison.matches:
    print(f"Diff: {comparison.diff}")

## 5. Regression Gate

Use regression gates to block a deployment if RAG quality drops:

In [ ]:
from evalai_sdk import evaluate_regression, to_pass_gate
from evalai_sdk.regression import Baseline, BaselineTolerance

# Define a baseline (from a previous known-good run)
baseline = Baseline(
    name="rag-v1",
    score=0.85,
    metrics={"grounding": 0.9, "relevance": 0.8},
    tolerance=BaselineTolerance(score=0.05, metrics=0.1),
)

# Simulate current results
current = {"score": 0.82, "metrics": {"grounding": 0.88, "relevance": 0.78}}

report = evaluate_regression(current, baseline)
gate_passed = to_pass_gate(report)

print(f"Gate passed: {gate_passed}")
print(f"Overall delta: {report.overall_delta.delta:+.2f}")
for d in report.metric_deltas:
    print(f"  {d.metric}: {d.delta:+.2f} (threshold: {d.threshold})")